# Expense Splitter Agent 

It uses OpenAI **tool calling** to:
- parse a pasted receipt (`parse_receipt`)
- split the bill deterministically (`split_bill`)
- export settlements to CSV (`export_transfers_csv`)

## Setup
- Ensure you have `OPENAI_API_KEY` in your repo-root `.env` file.
- Select the project venv kernel (like other labs).

In [ ]:
from __future__ import annotations

import json
import re
from decimal import Decimal, ROUND_HALF_UP
from typing import Any, Dict, List, Optional, Tuple

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai = OpenAI()


## Tools (deterministic Python functions)

These are the functions the model can call. They do all arithmetic and return JSON.

In [ ]:
def _to_decimal(x: Any) -> Decimal:
    if isinstance(x, Decimal):
        return x
    if isinstance(x, (int, float)):
        return Decimal(str(x))
    if isinstance(x, str):
        cleaned = x.strip().replace(",", "")
        cleaned = re.sub(r"[^0-9.\-]", "", cleaned)
        if cleaned in {"", "-", ".", "-."}:
            return Decimal("0")
        return Decimal(cleaned)
    return Decimal("0")


def _money(d: Decimal) -> Decimal:
    return d.quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)


def parse_receipt(text: str) -> Dict[str, Any]:
    lines = [ln.strip() for ln in (text or "").splitlines() if ln.strip()]
    items: List[Dict[str, Any]] = []

    for ln in lines:
        if re.search(r"\b(total|subtotal|tax|tip|gratuity|change|balance)\b", ln, re.I):
            continue

        qty = 1
        qty_m = re.search(r"(?i)\b(\d+)\s*x\b|\bx\s*(\d+)\b", ln)
        if qty_m:
            qty = int(qty_m.group(1) or qty_m.group(2))
            ln = re.sub(r"(?i)\b(\d+)\s*x\b|\bx\s*(\d+)\b", "", ln).strip()

        nums = list(re.finditer(r"[-+]?\d+(?:\.\d{1,2})?", ln.replace(",", "")))
        if not nums:
            continue

        price_token = nums[-1].group(0)
        price = _to_decimal(price_token)

        name_part = ln[: nums[-1].start()].strip(" -:\t")
        name_part = re.sub(r"\s{2,}", " ", name_part).strip()
        if not name_part:
            name_part = "Item"

        items.append(
            {
                "name": name_part,
                "qty": qty,
                "unit_price": float(_money(price)),
                "line_total": float(_money(price * qty)),
                "raw": ln,
            }
        )

    return {"items": items}


def _settle_transfers(net: Dict[str, Decimal]) -> List[Dict[str, Any]]:
    creditors: List[Tuple[str, Decimal]] = sorted(
        [(p, amt) for p, amt in net.items() if amt > 0], key=lambda x: x[1], reverse=True
    )
    debtors: List[Tuple[str, Decimal]] = sorted(
        [(p, -amt) for p, amt in net.items() if amt < 0], key=lambda x: x[1], reverse=True
    )

    transfers: List[Dict[str, Any]] = []
    ci = 0
    di = 0
    while ci < len(creditors) and di < len(debtors):
        c_name, c_amt = creditors[ci]
        d_name, d_amt = debtors[di]
        amt = _money(min(c_amt, d_amt))
        if amt > 0:
            transfers.append({"from": d_name, "to": c_name, "amount": float(amt)})
        c_amt = _money(c_amt - amt)
        d_amt = _money(d_amt - amt)
        creditors[ci] = (c_name, c_amt)
        debtors[di] = (d_name, d_amt)
        if c_amt == 0:
            ci += 1
        if d_amt == 0:
            di += 1
    return transfers


def split_bill(
    items: List[Dict[str, Any]],
    people: List[str],
    allocations: Optional[Dict[str, Any]] = None,
    tax: float = 0.0,
    tip: float = 0.0,
    fees: float = 0.0,
    payments: Optional[Dict[str, float]] = None,
) -> Dict[str, Any]:
    people = [p.strip() for p in (people or []) if p and p.strip()]
    if not people:
        return {"error": "people list is empty"}

    allocations = allocations or {}
    payments = payments or {}

    subtotals: Dict[str, Decimal] = {p: Decimal("0") for p in people}
    item_subtotal = Decimal("0")

    for idx, item in enumerate(items or []):
        line_total = _to_decimal(item.get("line_total", 0))
        if line_total <= 0:
            continue
        item_subtotal += line_total

        key = str(idx)
        rule = allocations.get(key)
        if rule is None:
            shares = {p: Decimal("1") for p in people}
        elif isinstance(rule, list):
            chosen = [p for p in rule if isinstance(p, str) and p in people]
            if not chosen:
                chosen = people
            shares = {p: (Decimal("1") if p in chosen else Decimal("0")) for p in people}
        elif isinstance(rule, dict):
            shares = {p: _to_decimal(rule.get(p, 0)) for p in people}
        else:
            shares = {p: Decimal("1") for p in people}

        total_share = sum(shares.values())
        if total_share == 0:
            shares = {p: Decimal("1") for p in people}
            total_share = Decimal(len(people))

        for p in people:
            part = (line_total * shares[p]) / total_share
            subtotals[p] += part

    extra = _money(_to_decimal(tax) + _to_decimal(tip) + _to_decimal(fees))

    owed: Dict[str, Decimal] = {p: Decimal("0") for p in people}
    if item_subtotal > 0:
        for p in people:
            owed[p] = _money(subtotals[p] + (extra * (subtotals[p] / item_subtotal)))
    else:
        per = _money(extra / Decimal(len(people)))
        for p in people:
            owed[p] = per

    total_due = _money(sum(owed.values()))

    if not payments:
        payments = {people[0]: float(total_due)}

    paid: Dict[str, Decimal] = {p: _money(_to_decimal(payments.get(p, 0))) for p in people}
    total_paid = _money(sum(paid.values()))

    if total_paid != total_due and total_paid > 0:
        scale = total_due / total_paid
        paid = {p: _money(paid[p] * scale) for p in people}
        total_paid = _money(sum(paid.values()))

    net: Dict[str, Decimal] = {p: _money(paid[p] - owed[p]) for p in people}
    transfers: List[Dict[str, Any]] = _settle_transfers(net)

    return {
        "people": people,
        "per_person": {
            p: {
                "subtotal": float(_money(subtotals[p])),
                "owed": float(owed[p]),
                "paid": float(paid[p]),
                "net": float(net[p]),
            }
            for p in people
        },
        "totals": {
            "items_subtotal": float(_money(item_subtotal)),
            "extras": float(extra),
            "total_due": float(_money(sum(owed.values()))),
        },
        "transfers": transfers,
    }


def export_transfers_csv(transfers: List[Dict[str, Any]]) -> Dict[str, Any]:
    rows = ["from,to,amount"]
    for t in transfers or []:
        rows.append(f"{t.get('from','')},{t.get('to','')},{t.get('amount',0)}")
    return {"csv": "\n".join(rows)}


## Tool schemas + tool-call loop

This is the same pattern used in `4_lab4.ipynb` (while-loop that keeps running tool calls until the model returns a final answer).

In [ ]:
parse_receipt_json = {
    "name": "parse_receipt",
    "description": "Parse a pasted receipt into structured items (name, qty, unit_price, line_total).",
    "parameters": {
        "type": "object",
        "properties": {"text": {"type": "string", "description": "Receipt text pasted by the user."}},
        "required": ["text"],
        "additionalProperties": False,
    },
}

split_bill_json = {
    "name": "split_bill",
    "description": "Split a bill across people based on receipt items and allocations, and return who should pay whom.",
    "parameters": {
        "type": "object",
        "properties": {
            "items": {"type": "array", "description": "Array of receipt items from parse_receipt().", "items": {"type": "object"}},
            "people": {"type": "array", "description": "List of participant names.", "items": {"type": "string"}},
            "allocations": {"type": "object", "description": "Mapping item_index (string) -> list of people OR mapping person->fraction. Unspecified items default to everyone."},
            "tax": {"type": "number", "description": "Tax amount (not percent)."},
            "tip": {"type": "number", "description": "Tip amount (not percent)."},
            "fees": {"type": "number", "description": "Other fees amount."},
            "payments": {"type": "object", "description": "Mapping person -> amount paid. If empty, assumes first person paid the total."},
        },
        "required": ["items", "people"],
        "additionalProperties": False,
    },
}

export_transfers_csv_json = {
    "name": "export_transfers_csv",
    "description": "Export computed transfers to CSV (from,to,amount).",
    "parameters": {
        "type": "object",
        "properties": {"transfers": {"type": "array", "description": "Transfers list from split_bill().", "items": {"type": "object"}}},
        "required": ["transfers"],
        "additionalProperties": False,
    },
}

tools = [
    {"type": "function", "function": parse_receipt_json},
    {"type": "function", "function": split_bill_json},
    {"type": "function", "function": export_transfers_csv_json},
]

TOOL_FUNCTIONS = {
    "parse_receipt": parse_receipt,
    "split_bill": split_bill,
    "export_transfers_csv": export_transfers_csv,
}


def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments or "{}")
        tool = TOOL_FUNCTIONS.get(tool_name)
        result = tool(**arguments) if tool else {"error": f"Unknown tool: {tool_name}"}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results


## Gradio chat UI

Run the next cell to launch the chat interface.

In [ ]:
system_prompt = """You are an expense-splitting assistant.

You can:
- parse receipts pasted as text using parse_receipt
- split a bill across people using split_bill
- export settlements to CSV using export_transfers_csv

Rules:
- Ask for the participant names first if missing.
- If allocations are unclear, ask a short clarifying question OR default to splitting those items across everyone.
- Prefer calling tools instead of doing arithmetic in your head.
- When you present results, show a simple "who pays whom" list and also per-person totals.
"""


def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        if response.choices[0].finish_reason == "tool_calls":
            msg = response.choices[0].message
            results = handle_tool_calls(msg.tool_calls)
            messages.append(msg)
            messages.extend(results)
        else:
            done = True
    return response.choices[0].message.content


gr.ChatInterface(chat, type="messages", title="Expense Splitter Agent (Tool Calling)").launch()
